# 05 — Improving Retrieval

This notebook compares retrieval strategies: keyword (BM25), semantic (embedding), hybrid (RRF fusion), and reranking (cross-encoder).

**What you'll learn:**
- How BM25 keyword search works and where it beats semantic search
- Reciprocal Rank Fusion for combining ranked lists
- Cross-encoder reranking for maximum precision
- When to use each strategy

**No API key needed** — everything runs locally.

## 1. Setup

In [ ]:
import sys, os
os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, 'src')

from rag_pipeline.retriever import HybridRetriever, BM25, rerank

print('Modules loaded successfully')

## 2. Index the documents

The `HybridRetriever` builds **both** a ChromaDB vector index (for semantic search) and a BM25 index (for keyword search) from the same chunks.

In [ ]:
retriever = HybridRetriever(persist_dir='./chroma_db')
result = retriever.index('data/sample/')

print('Indexing complete:')
for key, val in result.items():
    print(f'  {key}: {val}')

stats = retriever.stats()
print(f'\nBM25 vocabulary size: {stats["bm25_vocab_size"]} unique terms')

## 3. Keyword vs. semantic — where each wins

Let's test queries that expose the strengths and weaknesses of each approach.

In [ ]:
test_queries = [
    # Natural language — semantic should win
    'Can I work from home?',
    # Exact term — keyword should win
    '401k',
    # Paraphrase — semantic should win
    'company savings plan for retirement',
    # Acronym + context — hybrid should win
    'FMLA leave eligibility',
]

for query in test_queries:
    print(f'\n{"=" * 60}')
    print(f'Query: "{query}"')
    print(f'{"-" * 60}')
    
    for mode in ['semantic', 'keyword', 'hybrid']:
        results = retriever.retrieve(query, mode=mode, top_k=1)
        if results:
            r = results[0]
            source = r['metadata']['source'].split('/')[-1]
            print(f'  {mode:<10} [{r["score"]:.4f}] {source} | {r["document"][:70]}...')
        else:
            print(f'  {mode:<10} [no results]')

## 4. BM25 internals — how keyword scoring works

Let's peek inside BM25 to see term frequency and IDF in action.

In [ ]:
bm25 = retriever.bm25

# Show some IDF values — rare terms have higher IDF
sample_terms = ['the', 'policy', '401k', 'fmla', 'canary', 'deployment']
print('Term IDF values (higher = rarer = more informative):')
print(f'{"Term":<15} {"Doc Freq":>10} {"IDF":>10}')
print('-' * 37)
for term in sample_terms:
    df = bm25.doc_freqs.get(term, 0)
    idf = bm25._idf(term)
    print(f'{term:<15} {df:>10} {idf:>10.3f}')

print(f'\nTotal documents in BM25 index: {bm25.doc_count}')
print(f'Average document length: {bm25.avg_dl:.0f} tokens')

## 5. Hybrid search with RRF fusion

Hybrid search runs both keyword and semantic, then merges with Reciprocal Rank Fusion. Let's see the fusion in action.

In [ ]:
question = '401k retirement matching policy'

# Run each mode separately to see what gets fused
sem_results = retriever.retrieve(question, mode='semantic', top_k=5)
kw_results = retriever.retrieve(question, mode='keyword', top_k=5)
hybrid_results = retriever.retrieve(question, mode='hybrid', top_k=5)

print('SEMANTIC top 5:')
for i, r in enumerate(sem_results, 1):
    print(f'  #{i} [{r["score"]:.4f}] {r["id"]} | {r["document"][:60]}...')

print('\nKEYWORD top 5:')
for i, r in enumerate(kw_results, 1):
    print(f'  #{i} [{r["score"]:.4f}] {r["id"]} | {r["document"][:60]}...')

print('\nHYBRID (RRF fused) top 5:')
for i, r in enumerate(hybrid_results, 1):
    print(f'  #{i} [{r["score"]:.4f}] {r["id"]} | {r["document"][:60]}...')

# Show which docs appear in both vs only one
sem_ids = {r['id'] for r in sem_results}
kw_ids = {r['id'] for r in kw_results}
print(f'\nIn both:         {sem_ids & kw_ids}')
print(f'Semantic only:   {sem_ids - kw_ids}')
print(f'Keyword only:    {kw_ids - sem_ids}')

## 6. Reranking with a cross-encoder

The cross-encoder evaluates each (query, document) pair together for maximum precision. The first time you run this, it downloads a small model (~22MB).

In [ ]:
question = 'What is the code review process?'

# Without reranking
no_rerank = retriever.retrieve(question, mode='hybrid', top_k=5)

# With reranking (retrieves 15 candidates, reranks to top 5)
with_rerank = retriever.retrieve(
    question, mode='hybrid', top_k=5, use_rerank=True
)

print('WITHOUT reranking:')
for i, r in enumerate(no_rerank, 1):
    print(f'  #{i} [{r["score"]:.4f}] ({r["method"]}) {r["document"][:70]}...')

print('\nWITH reranking:')
for i, r in enumerate(with_rerank, 1):
    print(f'  #{i} [{r["score"]:.4f}] ({r["method"]}) {r["document"][:70]}...')

# Did the order change?
order_before = [r['id'] for r in no_rerank]
order_after = [r['id'] for r in with_rerank]
print(f'\nOrder changed: {order_before != order_after}')

## 7. Systematic comparison across query types

Let's test a range of queries and see which mode places the most relevant chunk at #1.

In [ ]:
test_cases = [
    ('Can I work from home?', 'remote'),
    ('401k', '401'),
    ('company savings plan for retirement', '401'),
    ('FMLA leave eligibility', 'fmla'),
    ('What happens during deployment?', 'deploy'),
    ('How do I set up my development environment?', 'development environment'),
    ('PTO accrual rate', 'pto'),
    ('canary deployment percentage', 'canary'),
]

modes = ['semantic', 'keyword', 'hybrid']
wins = {m: 0 for m in modes}

print(f'{"Query":<45} ', end='')
for m in modes:
    print(f'{m:<10}', end='')
print()
print('-' * 75)

for query, target in test_cases:
    print(f'{query:<45} ', end='')
    for mode in modes:
        results = retriever.retrieve(query, mode=mode, top_k=1)
        if results and target.lower() in results[0]['document'].lower():
            print(f'{"  HIT":<10}', end='')
            wins[mode] += 1
        else:
            print(f'{"  miss":<10}', end='')
    print()

print('-' * 75)
print(f'{"TOTAL HITS":<45} ', end='')
for mode in modes:
    print(f'  {wins[mode]}/{len(test_cases):<7}', end='')
print()

## 8. Latency comparison

Speed matters in production. Let's measure each mode.

In [ ]:
import time

question = 'What is the remote work policy?'
modes_to_test = [
    ('semantic', False),
    ('keyword', False),
    ('hybrid', False),
    ('hybrid+rerank', True),
]

print(f'{"Mode":<20} {"Latency (ms)":>12}')
print('-' * 34)

for label, use_rerank in modes_to_test:
    mode = 'hybrid' if 'hybrid' in label else label
    
    # Warm up
    retriever.retrieve(question, mode=mode, top_k=5, use_rerank=use_rerank)
    
    # Measure
    times = []
    for _ in range(5):
        start = time.perf_counter()
        retriever.retrieve(question, mode=mode, top_k=5, use_rerank=use_rerank)
        times.append((time.perf_counter() - start) * 1000)
    
    avg_ms = sum(times) / len(times)
    print(f'{label:<20} {avg_ms:>10.1f}ms')

## Key Takeaways

1. **Keyword search (BM25)** excels at exact terms, codes, and proper nouns
2. **Semantic search** excels at natural language questions and paraphrases
3. **Hybrid search (RRF)** combines both — the production default
4. **Reranking** with a cross-encoder adds ~100ms but improves precision significantly
5. **Retrieve-then-rerank** is the two-stage pattern: fast retrieval (broad), then precise reranking (narrow)

**Next:** We'll learn how to *measure* retrieval quality so you can prove your improvements actually work.